# MaaS Rate Limit & Policy Enforcement Test

This notebook tests MaaS **token-based rate limiting** end-to-end:

1. **Create Low-Limit Subscription** — 200 tokens/min for easy 429 trigger
2. **Rate Limit Trigger** — Send requests until Limitador returns 429
3. **Recovery** — Wait for the window to reset and confirm access restored
4. **Cleanup** — Remove test resources

> **Note:** Baseline inference and auth-rejection tests (401/403) are covered in
> `../2_maas/3_test_model_serving.ipynb`. Run that notebook first to confirm
> normal inference works before proceeding here.

**Prerequisites:**
- MaaS enabled with model registered (`../2_maas/2_enable_maas.ipynb`)
- Model inference verified (`../2_maas/3_test_model_serving.ipynb`)
- MaaSSubscription configured with token rate limits

In [9]:
import subprocess, json, time, os
import urllib.request, ssl
from dotenv import load_dotenv

load_dotenv("../.env")

CLUSTER_DOMAIN = os.getenv("CLUSTER_DOMAIN")
if not CLUSTER_DOMAIN:
    r = subprocess.run(["oc", "get", "ingresses.config.openshift.io", "cluster",
                        "-o", "jsonpath={.spec.domain}"], capture_output=True, text=True)
    CLUSTER_DOMAIN = r.stdout.strip()

MODEL_NAMESPACE = os.getenv("MODEL_NAMESPACE", "demo")
MODEL_NAME = os.getenv("MODEL_NAME", "qwen36-27b")

MAAS_HOST = f"https://maas-api.{CLUSTER_DOMAIN}"
INFERENCE_GW = os.getenv("MODEL_ENDPOINT", f"https://maas-api.{CLUSTER_DOMAIN}/{MODEL_NAMESPACE}/{MODEL_NAME}")

token_result = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True)
OC_TOKEN = token_result.stdout.strip()

API_KEY = os.getenv("MAAS_API_KEY", OC_TOKEN)

ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

print(f"Inference GW: {INFERENCE_GW}")
print(f"Model:        {MODEL_NAME}")
print(f"Auth:         {'API key' if os.getenv('MAAS_API_KEY') else 'OCP token'}")

Inference GW: https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/demo/qwen36-27b
Model:        qwen36-27b
Auth:         API key


## 1. Create Low-Limit Subscription for Testing

![subscription](../images/subscription.png)

\
Create a subscription with a very low rate limit (**50 tokens/min**, priority 50)
and bind an API key to it. The higher priority overrides the default
`lab-subscription` (10000 tok/min, priority 10).

> **Note:** `kube:admin` contains a colon, so it cannot be added to OpenShift
> groups via `oc adm groups add-users`. We use the `users` field directly instead.

In [10]:
%%bash
source ../.env 2>/dev/null || true
MODEL_NS=${MODEL_NAMESPACE:-demo}
OCP_USER=$(oc whoami)
MODEL_REF=$(oc get maasmodelref -n ${MODEL_NS} -o jsonpath='{.items[0].metadata.name}' 2>/dev/null)

if [ -z "$MODEL_REF" ]; then
    echo "ERROR: No MaaSModelRef found in namespace '${MODEL_NS}'."
    exit 1
fi

echo "Creating low-limit subscription for user: ${OCP_USER}"
echo "  Model ref: ${MODEL_NS}/${MODEL_REF}"
echo ""

oc apply -f - <<EOF
apiVersion: maas.opendatahub.io/v1alpha1
kind: MaaSAuthPolicy
metadata:
  name: lab-ratelimit-test
  namespace: models-as-a-service
spec:
  modelRefs:
    - name: ${MODEL_REF}
      namespace: ${MODEL_NS}
  subjects:
    groups: []
    users:
      - "${OCP_USER}"
EOF

oc apply -f - <<EOF
apiVersion: maas.opendatahub.io/v1alpha1
kind: MaaSSubscription
metadata:
  name: lab-ratelimit-test
  namespace: models-as-a-service
spec:
  owner:
    groups: []
    users:
      - "${OCP_USER}"
  modelRefs:
    - name: ${MODEL_REF}
      namespace: ${MODEL_NS}
      tokenRateLimits:
        - limit: 50
          window: 1m
  priority: 50
EOF

echo ""
echo "Waiting for reconciliation (30s)..."
sleep 30
echo "Done."
oc get maassubscription -n models-as-a-service --no-headers

Creating low-limit subscription for user: kube:admin
  Model ref: demo/qwen36-27b

maasauthpolicy.maas.opendatahub.io/lab-ratelimit-test created
maassubscription.maas.opendatahub.io/lab-ratelimit-test created

Waiting for reconciliation (30s)...
Done.
lab-ratelimit-test   Active   50    30s
lab-subscription     Active   10    4d5h


### 1.1 Create API Key Bound to Low-Limit Subscription

![API Key](../images/api_key.png)

In [11]:
key_data = json.dumps({
    "name": "ratelimit-test-key",
    "subscription": "lab-ratelimit-test",
    "expiresIn": "1h"
}).encode()

req = urllib.request.Request(
    f"{MAAS_HOST}/maas-api/v1/api-keys",
    data=key_data,
    headers={"Authorization": f"Bearer {OC_TOKEN}", "Content-Type": "application/json"},
    method="POST"
)

try:
    with urllib.request.urlopen(req, context=ctx, timeout=15) as resp:
        result = json.loads(resp.read())
    TEST_KEY = result.get("key", "")
    print(f"Test key created: {TEST_KEY[:20]}...")
    print(f"  Subscription: {result.get('subscription', '')}")
    print(f"  Limit: 50 tokens/min (priority 50)")
except urllib.error.HTTPError as e:
    err = e.read().decode()[:200] if e.fp else ""
    print(f"ERROR: HTTP {e.code} — {err}")
    print("Falling back to MAAS_API_KEY (rate limit test may not trigger 429)")
    TEST_KEY = API_KEY
except Exception as e:
    print(f"ERROR: {e}")
    TEST_KEY = API_KEY

Test key created: sk-oai-BATvQOV0RLpU2...
  Subscription: lab-ratelimit-test
  Limit: 50 tokens/min (priority 50)


## 2. Trigger Rate Limit (429)

Send requests with `max_tokens=30` to consume the 50 tok/min quota in 2-3 requests.

In [4]:
def request_with_key(key, prompt, max_tokens=30):
    body = json.dumps({
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens
    }).encode()
    req = urllib.request.Request(
        f"{INFERENCE_GW}/v1/chat/completions",
        data=body,
        headers={"Authorization": f"Bearer {key}", "Content-Type": "application/json"},
        method="POST"
    )
    try:
        with urllib.request.urlopen(req, context=ctx, timeout=30) as resp:
            data = json.loads(resp.read())
            tokens_used = data.get("usage", {}).get("total_tokens", 0)
            return resp.status, tokens_used
    except urllib.error.HTTPError as e:
        return e.code, 0
    except Exception as e:
        return 0, 0

print("Rate Limit Trigger Test (50 tokens/min limit, priority 50)")
print("=" * 60)
print("")

total_tokens = 0
got_429 = False

for i in range(10):
    status, tokens = request_with_key(TEST_KEY, "Say hello.", max_tokens=30)
    total_tokens += tokens
    
    if status == 429:
        print(f"  Request {i+1:>2}: HTTP 429 ← RATE LIMIT HIT  (total tokens: ~{total_tokens})")
        got_429 = True
        break
    elif status == 200:
        print(f"  Request {i+1:>2}: HTTP 200  (tokens this req: {tokens}, total: ~{total_tokens})")
    else:
        print(f"  Request {i+1:>2}: HTTP {status}")
        break

print("")
if got_429:
    print(f"  SUCCESS: Rate limit enforced after ~{total_tokens} tokens")
else:
    print(f"  NOTE: No 429 received. Used ~{total_tokens} tokens.")
    print(f"  The subscription may not have taken effect yet, or priority is not overriding.")

Rate Limit Trigger Test (50 tokens/min limit, priority 50)

  Request  1: HTTP 200  (tokens this req: 43, total: ~43)
  Request  2: HTTP 200  (tokens this req: 43, total: ~86)
  Request  3: HTTP 429 ← RATE LIMIT HIT  (total tokens: ~86)

  SUCCESS: Rate limit enforced after ~86 tokens


## 3. Recovery After Rate Limit Window Reset

Wait for the 1-minute window to elapse, then verify access is restored.

In [5]:
if got_429:
    print("Waiting 65 seconds for rate limit window to reset...")
    for remaining in range(65, 0, -10):
        print(f"  {remaining}s remaining...", flush=True)
        time.sleep(10)
    time.sleep(5)
    
    print("\nRetrying after window reset:")
    status, tokens = request_with_key(TEST_KEY, "Say OK.")
    if status == 200:
        print(f"  HTTP {status} — Access restored!")
        print(f"  PASS: Rate limit resets correctly after window expires.")
    elif status == 429:
        print(f"  HTTP 429 — Still rate limited. Window may be longer than 1 min.")
    else:
        print(f"  HTTP {status} — Unexpected status")
else:
    print("Skipped — no 429 was triggered in previous step.")

Waiting 65 seconds for rate limit window to reset...
  65s remaining...
  55s remaining...
  45s remaining...
  35s remaining...
  25s remaining...
  15s remaining...
  5s remaining...

Retrying after window reset:
  HTTP 200 — Access restored!
  PASS: Rate limit resets correctly after window expires.


## 4. Cleanup

Removes test resources created in both `1_maas_advanced.ipynb` and this notebook.
The base `lab-subscription` and `MAAS_API_KEY` are preserved.

In [6]:
%%bash
echo "=== Cleaning up 1_maas_advanced.ipynb resources ==="
oc delete maassubscription lab-free-subscription -n models-as-a-service 2>/dev/null && echo "  Deleted lab-free-subscription" || echo "  (not found)"
oc delete maassubscription lab-premium-subscription -n models-as-a-service 2>/dev/null && echo "  Deleted lab-premium-subscription" || echo "  (not found)"
oc delete maasauthpolicy lab-free-access -n models-as-a-service 2>/dev/null && echo "  Deleted lab-free-access" || echo "  (not found)"
oc delete maasauthpolicy lab-premium-access -n models-as-a-service 2>/dev/null && echo "  Deleted lab-premium-access" || echo "  (not found)"

echo ""
echo "=== Cleaning up 2_maas_policy_test.ipynb resources ==="
oc delete maassubscription lab-ratelimit-test -n models-as-a-service 2>/dev/null && echo "  Deleted lab-ratelimit-test subscription" || echo "  (not found)"
oc delete maasauthpolicy lab-ratelimit-test -n models-as-a-service 2>/dev/null && echo "  Deleted lab-ratelimit-test auth policy" || echo "  (not found)"

echo ""
echo "Cleanup complete. Base lab-subscription and MAAS_API_KEY are preserved."

=== Cleaning up 1_maas_advanced.ipynb resources ===
  (not found)
  (not found)
  (not found)
  (not found)

=== Cleaning up 2_maas_policy_test.ipynb resources ===
maassubscription.maas.opendatahub.io "lab-ratelimit-test" deleted from models-as-a-service namespace
  Deleted lab-ratelimit-test subscription
maasauthpolicy.maas.opendatahub.io "lab-ratelimit-test" deleted from models-as-a-service namespace
  Deleted lab-ratelimit-test auth policy

Cleanup complete. Base lab-subscription and MAAS_API_KEY are preserved.


In [7]:
import urllib.request, json

test_key_names = {"free-tier-test", "premium-tier-test", "ratelimit-test-key"}

search_data = json.dumps({"status": "active", "limit": 50, "includeEphemeral": True}).encode()
req = urllib.request.Request(
    f"{MAAS_HOST}/maas-api/v1/api-keys/search",
    data=search_data,
    headers={"Authorization": f"Bearer {OC_TOKEN}", "Content-Type": "application/json"},
    method="POST"
)

revoked = 0
try:
    with urllib.request.urlopen(req, context=ctx, timeout=15) as resp:
        keys_list = json.loads(resp.read())
    items = keys_list if isinstance(keys_list, list) else keys_list.get("items", keys_list.get("data", []))
    for key in items:
        if key.get("name") in test_key_names:
            key_id = key.get("id", "")
            del_req = urllib.request.Request(
                f"{MAAS_HOST}/maas-api/v1/api-keys/{key_id}",
                headers={"Authorization": f"Bearer {OC_TOKEN}"},
                method="DELETE"
            )
            try:
                with urllib.request.urlopen(del_req, context=ctx, timeout=10):
                    print(f"  Revoked: {key.get('name')} (id={key_id[:12]}...)")
                    revoked += 1
            except urllib.error.HTTPError as e:
                print(f"  Failed to revoke {key.get('name')}: HTTP {e.code}")
except Exception as e:
    print(f"  Note: {e}")

print(f"\nRevoked {revoked} test key(s). Main MAAS_API_KEY is preserved.")

  Revoked: ratelimit-test-key (id=993eee3b-da8...)
  Revoked: ratelimit-test-key (id=8b788acb-fa0...)

Revoked 2 test key(s). Main MAAS_API_KEY is preserved.


---
## Summary

| Test | Result | What It Proves |
|------|--------|----------------|
| Rate Limit Trigger | HTTP 429 | Limitador enforces token quotas |
| Recovery | HTTP 200 after 60s | Rate limit windows reset correctly |

> Baseline inference and auth-rejection tests (401/403) are in
> `../2_maas/3_test_model_serving.ipynb`.

### How Rate Limiting Works in MaaS

```
Request → Gateway → Authorino (auth) → Limitador (rate limit check) → Backend
                                              ↓
                                    429 if over quota
```

- **MaaSSubscription** defines the quota (tokens per window per group)
- **Limitador** (from RHCL) enforces it at the gateway level
- **Window-based reset**: Once the window (e.g., 1 min) expires, the counter resets

### Next Steps

- `1_maas_advanced.ipynb` — Multi-tier subscriptions, observability
- Return to `../2_maas/2_enable_maas.ipynb` to adjust global subscription quotas